# Calibration Edge Cases Analysis

This notebook utilizes calibration data to identify and analyze specific edge cases, such as morning temperature drops and Domestic Hot Water (DHW) overshoots, to improve model robustness.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta, timezone

from src.analysis.data_loader import DataLoader

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_context("notebook")
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Data Loading

Load high-resolution calibration data focusing on periods known for edge cases.

In [ ]:
loader = DataLoader()
end_time = datetime.now(timezone.utc)
start_time = end_time - timedelta(days=7)

features = [
    'indoor_temperature',
    'outdoor_temperature',
    'ml_target_temperature',
    'heating_state',
    'dhw_active',
    'prediction_error'
]

try:
    df = loader.fetch_training_data(start_time=start_time, end_time=end_time, features=features)
    print(f"Loaded {len(df)} records from {start_time.strftime('%Y-%m-%d')} to {end_time.strftime('%Y-%m-%d')}")
except Exception as e:
    print(f"Error loading data: {e}")
    # Create dummy data for demonstration
    dates = pd.date_range(start=start_time, end=end_time, freq='5min')
    df = pd.DataFrame(index=dates)
    
    # Simulate morning drop
    hour = pd.to_datetime(df.index).hour
    morning_drop = np.where((hour >= 5) & (hour <= 7), -0.5, 0)
    
    # Simulate DHW overshoot
    dhw_active = np.where((hour == 18) | (hour == 19), 1, 0)
    dhw_overshoot = np.where(dhw_active == 1, 0.8, 0)
    
    df['indoor_temperature'] = 20 + np.sin(np.linspace(0, 10, len(dates))) + morning_drop + dhw_overshoot + np.random.normal(0, 0.1, len(dates))
    df['ml_target_temperature'] = 21
    df['outdoor_temperature'] = 5 + np.sin(np.linspace(0, 10, len(dates))) * 5
    df['heating_state'] = np.where(df['indoor_temperature'] < 20.5, 1, 0)
    df['dhw_active'] = dhw_active
    df['prediction_error'] = df['indoor_temperature'] - df['ml_target_temperature']
    
    print("Using generated dummy data for demonstration.")

## 2. Morning Drop Analysis

Identify and analyze sudden temperature drops typically occurring in the early morning.

In [ ]:
if 'indoor_temperature' in df.columns:
    # Filter for morning hours (e.g., 4 AM to 8 AM)
    morning_data = df[(pd.to_datetime(df.index).hour >= 4) & (pd.to_datetime(df.index).hour <= 8)]
    
    # Calculate temperature change rate
    morning_data['temp_change_rate'] = morning_data['indoor_temperature'].diff()
    
    # Identify significant drops
    significant_drops = morning_data[morning_data['temp_change_rate'] < -0.2]
    
    plt.figure(figsize=(14, 6))
    plt.plot(morning_data.index, morning_data['indoor_temperature'], label='Indoor Temp (Morning)')
    plt.scatter(significant_drops.index, significant_drops['indoor_temperature'], color='red', label='Significant Drop', zorder=5)
    plt.title('Morning Temperature Drops Analysis')
    plt.xlabel('Time')
    plt.ylabel('Temperature (°C)')
    plt.legend()
    plt.show()
    
    print(f"Identified {len(significant_drops)} significant morning drops.")
else:
    print("Indoor temperature data not available.")

## 3. DHW Overshoot Analysis

Analyze temperature overshoots correlated with Domestic Hot Water (DHW) heating cycles.

In [ ]:
if 'dhw_active' in df.columns and 'indoor_temperature' in df.columns:
    fig, ax1 = plt.subplots(figsize=(14, 6))
    
    color = 'tab:red'
    ax1.set_xlabel('Time')
    ax1.set_ylabel('Indoor Temperature (°C)', color=color)
    ax1.plot(df.index, df['indoor_temperature'], color=color, label='Indoor Temp')
    ax1.plot(df.index, df['ml_target_temperature'], color='tab:orange', linestyle='--', label='Target Temp')
    ax1.tick_params(axis='y', labelcolor=color)
    ax1.legend(loc='upper left')
    
    ax2 = ax1.twinx()  
    color = 'tab:blue'
    ax2.set_ylabel('DHW Active', color=color)  
    ax2.fill_between(df.index, 0, df['dhw_active'], color=color, alpha=0.3, step='post', label='DHW Active')
    ax2.tick_params(axis='y', labelcolor=color)
    ax2.set_yticks([0, 1])
    ax2.set_yticklabels(['OFF', 'ON'])
    
    fig.tight_layout()  
    plt.title('Indoor Temperature vs DHW Activity (Overshoot Analysis)')
    plt.show()
    
    # Calculate average overshoot during DHW active periods
    dhw_on_data = df[df['dhw_active'] == 1]
    if not dhw_on_data.empty:
        avg_overshoot = (dhw_on_data['indoor_temperature'] - dhw_on_data['ml_target_temperature']).mean()
        print(f"Average temperature deviation during DHW cycles: {avg_overshoot:.2f} °C")
else:
    print("DHW activity or indoor temperature data not available.")